[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/pandera-certified/notebooks/day-05-function-decorators.ipynb#scrollTo=aa11bb22)

---
# Day 5 · Function Decorators — @pa.check_input and @pa.check_output
**certified-journeys / pandera-certified** · Automatic DataFrame validation at function boundaries

> **Goal for today:** Attach input and output schema checks to pipeline functions using decorators so validation is automatic, invisible to the caller, and easy to test.


In [ ]:
%pip install -q pandera


## Step 1 · How `@pa.check_input` works

`check_input(schema)` wraps a function so that the **first positional argument** (or the argument specified by `obj_getter`) is validated against `schema` before the function body runs.

The decorator signature:
```python
pa.check_input(schema, obj_getter=None)
```

- `schema` — a `DataFrameSchema` or `DataFrameModel` to validate against.
- `obj_getter` — optional `int` or `str` that selects which positional/keyword argument to validate (default: 0, the first).

If the input DataFrame violates the schema a `SchemaError` is raised **before the function body executes** — preventing corrupt data from propagating downstream.


In [ ]:
import pandas as pd
import pandera as pa
from pandera import check_input, check_output

# Schema that the cleaning function expects as input
RawSchema = pa.DataFrameSchema(
    {
        "temperature": pa.Column(float, pa.Check.between(-50.0, 60.0), nullable=False),
        "humidity": pa.Column(float, pa.Check.between(0.0, 100.0), nullable=False),
    },
    name="RawSchema",
)


@check_input(RawSchema)
def clean_readings(df: pd.DataFrame) -> pd.DataFrame:
    """Round sensor readings to 1 decimal place."""
    return df.round(1)


# --- Happy path ---
valid_raw = pd.DataFrame(
    {"temperature": [22.456, 18.7], "humidity": [55.0, 62.3]}
)
result = clean_readings(valid_raw)
print("Clean output (happy path):")
print(result)

# --- Sad path: pass an invalid DataFrame ---
bad_raw = pd.DataFrame(
    {"temperature": [200.0, 18.7], "humidity": [55.0, 62.3]}  # 200 > 60 → invalid
)

try:
    clean_readings(bad_raw)
except pa.errors.SchemaError as exc:
    print("\n--- SchemaError on invalid input ---")
    print(f"Column  : {exc.schema.name}")
    print(f"Failures: {exc.failure_cases}")


**What just happened?**

- **`@check_input(RawSchema)`** intercepts the call before `clean_readings` body runs.
- The valid DataFrame passes through unchanged; the function then rounds values.
- The bad DataFrame (temperature 200 °C) raises `SchemaError` immediately — the function body never executes.
- **`exc.failure_cases`** is a DataFrame pinpointing exactly which row and value violated the check.


## Step 2 · Guard outputs with `@pa.check_output`

`check_output(schema)` validates the **return value** of a function against `schema`. This catches bugs where the function body itself produces invalid data — a category of error that input checks cannot catch.

Common use: ensure a transformation that computes derived columns produces the expected shape and value ranges before the result is passed to the next stage.


In [ ]:
import pandas as pd
import pandera as pa
from pandera import check_output

OutputSchema = pa.DataFrameSchema(
    {
        "temperature": pa.Column(float),
        "humidity": pa.Column(float),
        # The function must produce a 'feels_like' column between -50 and 60
        "feels_like": pa.Column(float, pa.Check.between(-50.0, 60.0)),
    },
    name="OutputSchema",
)


@check_output(OutputSchema)
def compute_feels_like(df: pd.DataFrame) -> pd.DataFrame:
    """Compute heat-index approximation (simplified formula)."""
    # NOTE: this simplified formula can produce values > 60 for high humidity
    df = df.copy()
    df["feels_like"] = df["temperature"] + (df["humidity"] - 40) * 0.1
    return df


# --- Correct output (feels_like stays in range) ---
normal_input = pd.DataFrame({"temperature": [22.0, 18.0], "humidity": [55.0, 40.0]})
print("Correct output:")
print(compute_feels_like(normal_input))

# --- Buggy output: high humidity pushes feels_like above 60 ---
extreme_input = pd.DataFrame({"temperature": [58.0], "humidity": [99.0]})

try:
    compute_feels_like(extreme_input)
except pa.errors.SchemaError as exc:
    print("\n--- Output SchemaError ---")
    print(f"Column  : {exc.schema.name}")
    print(f"Failures: {exc.failure_cases}")


**What just happened?**

- **`@check_output(OutputSchema)`** validates after the function body completes.
- The normal input produces a valid `feels_like` range and passes.
- The extreme input triggers the output check — the bug in our formula is caught **before** its result reaches downstream code.
- Output checks act like unit test assertions embedded in production code.


## Step 3 · Stacking `@check_input` and `@check_output` on the same function

Both decorators can be stacked. The execution order is:
1. `check_input` validates the argument.
2. Function body runs.
3. `check_output` validates the return value.

When stacking, apply `@check_output` first (outermost) so Python's decorator wrapping order results in `check_input` validating before the function and `check_output` validating after.


In [ ]:
import pandas as pd
import pandera as pa
from pandera import check_input, check_output

InSchema = pa.DataFrameSchema(
    {
        "value": pa.Column(float, pa.Check.gt(0.0)),  # strictly positive
    },
    name="InSchema",
)

OutSchema = pa.DataFrameSchema(
    {
        "value": pa.Column(float, pa.Check.gt(0.0)),
        "log_value": pa.Column(float),   # natural log of value
    },
    name="OutSchema",
)


@check_output(OutSchema)   # outermost → applied last at call time
@check_input(InSchema)     # innermost → applied first at call time
def add_log_column(df: pd.DataFrame) -> pd.DataFrame:
    import numpy as np
    return df.assign(log_value=np.log(df["value"]))


# Happy path
good = pd.DataFrame({"value": [1.0, 2.71828, 10.0]})
print("Stacked decorators — happy path:")
print(add_log_column(good))

# Sad path: negative input fails check_input BEFORE the function body
bad = pd.DataFrame({"value": [-1.0, 2.0]})
try:
    add_log_column(bad)
except pa.errors.SchemaError as exc:
    print(f"\nInput check caught: {exc.schema.name} → {exc.failure_cases['failure_case'].tolist()}")


**What just happened?**

- **Decorator order matters:** `@check_output` wraps `@check_input` which wraps the function — Python applies decorators bottom-up, so `check_input` fires first at call time.
- A negative `value` fails `check_input` and the function body never runs — `log` of a negative would have been `nan`, so the input check prevented a subtle data quality issue.
- Both schemas can coexist on a single function without any changes to the function signature.


## Step 4 · Selecting a specific argument with `obj_getter`

When a function accepts **multiple DataFrame arguments**, `obj_getter` tells `check_input` which one to validate:

- `obj_getter=0` (default) → first positional argument
- `obj_getter=1` → second positional argument
- `obj_getter="df_name"` → keyword argument named `df_name`

You can stack multiple `@check_input` decorators, each with a different `obj_getter`, to validate several arguments independently.


In [ ]:
import pandas as pd
import pandera as pa
from pandera import check_input

LeftSchema = pa.DataFrameSchema(
    {"id": pa.Column(int, pa.Check.ge(1)), "name": pa.Column(str)},
    name="LeftSchema",
)

RightSchema = pa.DataFrameSchema(
    {"id": pa.Column(int, pa.Check.ge(1)), "score": pa.Column(float, pa.Check.between(0, 100))},
    name="RightSchema",
)


@check_input(RightSchema, obj_getter=1)  # validate the SECOND arg (index 1)
@check_input(LeftSchema,  obj_getter=0)  # validate the FIRST arg  (index 0)
def merge_frames(left: pd.DataFrame, right: pd.DataFrame) -> pd.DataFrame:
    """Join left and right DataFrames on 'id'."""
    return left.merge(right, on="id")


left  = pd.DataFrame({"id": [1, 2, 3], "name": ["Alice", "Bob", "Carol"]})
right = pd.DataFrame({"id": [1, 2, 3], "score": [88.0, 75.5, 92.0]})
print("Merged result:")
print(merge_frames(left, right))

# Now pass a bad right DataFrame — score 150 is out of range
bad_right = pd.DataFrame({"id": [1, 2], "score": [88.0, 150.0]})
try:
    merge_frames(left, bad_right)
except pa.errors.SchemaError as exc:
    print(f"\nRight-side check caught: {exc.failure_cases}")


**What just happened?**

- `obj_getter=0` targets `left`, `obj_getter=1` targets `right` — both validated before the merge.
- The bad `right` DataFrame (score 150) fails before any join logic executes.
- You can stack as many `@check_input` decorators as there are DataFrame arguments in the function.
- Alternatively, use a keyword argument name as `obj_getter="right"` if you prefer named-argument targeting.


## Step 5 · `@pa.check_io` — validate inputs and outputs in one decorator

`pa.check_io` combines `check_input` and `check_output` in a single decorator call. The syntax:

```python
@pa.check_io(arg_name=InputSchema, out=OutputSchema)
def my_func(arg_name: pd.DataFrame) -> pd.DataFrame:
    ...
```

Key points:
- Keyword argument names match the **function parameter names**.
- `out` is the reserved keyword for the return value schema.
- Cleaner than stacking two decorators when you only have one input + one output.


In [ ]:
import pandas as pd
import pandera as pa

InSch = pa.DataFrameSchema(
    {"price": pa.Column(float, pa.Check.gt(0.0)), "qty": pa.Column(int, pa.Check.ge(1))},
    name="InSch",
)

OutSch = pa.DataFrameSchema(
    {
        "price": pa.Column(float, pa.Check.gt(0.0)),
        "qty": pa.Column(int),
        "revenue": pa.Column(float, pa.Check.gt(0.0)),  # price × qty must be > 0
    },
    name="OutSch",
)


@pa.check_io(df=InSch, out=OutSch)  # 'df' matches the parameter name
def compute_revenue(df: pd.DataFrame) -> pd.DataFrame:
    return df.assign(revenue=df["price"] * df["qty"])


orders = pd.DataFrame({"price": [9.99, 24.50, 4.99], "qty": [2, 1, 10]})
print("Revenue computation:")
print(compute_revenue(orders))

# Test: bad input (negative price)
bad_orders = pd.DataFrame({"price": [-1.0, 24.50], "qty": [2, 1]})
try:
    compute_revenue(bad_orders)
except pa.errors.SchemaError as exc:
    print(f"\ncheck_io input guard: {exc.failure_cases}")


**What just happened?**

- **`@pa.check_io(df=InSch, out=OutSch)`** maps argument name `df` to `InSch` and the return value to `OutSch`.
- One decorator replaces two stacked decorators — preferred when there's exactly one input DataFrame and one output.
- The `out` keyword is always reserved for the return value; any other keyword must match a function parameter name exactly.


## Step 6 · Writing a pytest-style test that asserts `SchemaError` is raised

Decorator-validated functions are easy to unit-test: pass invalid data and assert `pa.errors.SchemaError` is raised. This approach replaces brittle `if` guards with declarative schema assertions.

In a real test suite these would live in `test_pipeline.py` and run with `pytest`. Here we simulate the pattern inline.


In [ ]:
import pandas as pd
import pandera as pa
from pandera import check_input, check_output
import traceback

# ── The function under test ────────────────────────────────────────────────
NormSchema = pa.DataFrameSchema(
    {"x": pa.Column(float, pa.Check.between(0.0, 1.0))},
    name="NormSchema",
)


@check_input(NormSchema)
def scale_to_percent(df: pd.DataFrame) -> pd.DataFrame:
    return df.assign(x_pct=df["x"] * 100.0)


# ── pytest-style tests (no pytest needed; using assert + try/except) ───────
def test_valid_input_passes():
    valid = pd.DataFrame({"x": [0.0, 0.5, 1.0]})
    result = scale_to_percent(valid)
    assert "x_pct" in result.columns
    assert (result["x_pct"] >= 0.0).all() and (result["x_pct"] <= 100.0).all()
    print("PASS test_valid_input_passes")


def test_invalid_input_raises():
    """Assert that an out-of-range value triggers SchemaError."""
    invalid = pd.DataFrame({"x": [0.5, 1.5]})  # 1.5 > 1.0
    try:
        scale_to_percent(invalid)
        raise AssertionError("Expected SchemaError was NOT raised")
    except pa.errors.SchemaError as exc:
        # Verify the failure case contains the violating value
        assert 1.5 in exc.failure_cases["failure_case"].values
        print("PASS test_invalid_input_raises")


def test_missing_column_raises():
    """Assert that a DataFrame missing required columns triggers SchemaError."""
    no_x = pd.DataFrame({"y": [0.1, 0.2]})
    try:
        scale_to_percent(no_x)
        raise AssertionError("Expected SchemaError was NOT raised")
    except pa.errors.SchemaError:
        print("PASS test_missing_column_raises")


# Run all tests
test_valid_input_passes()
test_invalid_input_raises()
test_missing_column_raises()
print("\nAll tests passed.")


**What just happened?**

- Each test function follows the standard **Arrange → Act → Assert** pattern.
- `test_invalid_input_raises` uses a try/except pattern that will **itself raise** `AssertionError` if the `SchemaError` is not raised — mirroring `pytest.raises(pa.errors.SchemaError)`.
- **`exc.failure_cases["failure_case"].values`** lets you assert on the specific violating value, not just that an error was raised.
- In a real pytest suite: `with pytest.raises(pa.errors.SchemaError): scale_to_percent(invalid)` is the idiomatic form.


In [ ]:
# Challenge: Full decorated pipeline stage
#
# 1. Define two schemas:
#    RawOrderSchema — columns: order_id (int, >=1), units (int, >=1), unit_price (float, >0.0)
#    EnrichedOrderSchema — same columns plus: total (float, >0.0), category (str)
#
# 2. Write a function `enrich_orders(df, category)` decorated with @pa.check_io:
#    - df validates against RawOrderSchema
#    - return value validates against EnrichedOrderSchema
#    - body: add total = units * unit_price, add category column from the arg
#
# 3. Write two tests:
#    a) valid DataFrame + valid category → asserts 'total' column exists
#    b) negative unit_price → asserts SchemaError raised

# Your solution here


---
## Day 5 key concepts recap

| Concept | What to remember |
|---|---|
| `@check_input(schema)` | Validates argument 0 (or `obj_getter`) before function body runs |
| `@check_output(schema)` | Validates return value after function body runs |
| Stacking order | Apply `@check_output` above `@check_input`; Python applies bottom-up |
| `obj_getter` | `int` index or `str` keyword arg name to select which argument to validate |
| `@pa.check_io` | Combines input + output in one decorator; use keyword arg names |
| `out=` keyword | Reserved kwarg in `check_io` for the return value schema |
| Pytest pattern | `pytest.raises(pa.errors.SchemaError)` or try/except with re-raise |

> **Tip:** Prefer `@pa.check_io` for simple one-input / one-output functions; fall back to stacked `@check_input` / `@check_output` when you need different `obj_getter` values.

---
## What's next
**Day 6** → Integrate Hypothesis for property-based testing and use `pa.infer_schema()` to bootstrap schemas from existing DataFrames.

Mark Day 5 complete in your [tracker](../index.html).
